# 02 · 샘플 기술/노하우 데이터 주입

**파이프라인 순서 2/4** — 파이프라인을 테스트할 수 있도록 실제 Teams/Mail에
샘플 기술·노하우 메시지/메일을 **write 도구로 전송**합니다.

여기서는 MCP 툴을 직접 고르는 대신 **에이전트(LLM 도구호출 루프)** 에게 자연어로 지시합니다.
에이전트가 각 서버의 알맞은 write 툴(메일 전송 / Teams 메시지 게시)을 선택해 실행합니다.

> ⚠️ 실제 메일/메시지가 전송됩니다. 기본값은 **본인에게 보내기 / 본인과의 채팅**입니다.
> 아래 `MAIL_TARGET`, `TEAMS_TARGET`을 조직 정책에 맞게 수정하세요.

In [ ]:
# --- Bootstrap: locate llmwiki-pipeline/app and import the shared package ---
import sys
from pathlib import Path

def _find_app_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'app', base / 'llmwiki-pipeline' / 'app'):
            if (cand / 'pipeline' / '__init__.py').is_file():
                return cand
    raise RuntimeError('Could not locate llmwiki-pipeline/app — run this from inside the project.')

APP_DIR = _find_app_dir()
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
print('app dir on sys.path:', APP_DIR)

## 1. 토큰 확보 (01에서 캐시된 로그인 재사용)

In [ ]:
from pipeline import auth, config

tokens = {}
for key in config.SOURCE_KEYS:
    try:
        tokens[key] = auth.ensure_access_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')
print('확보된 소스:', list(tokens.keys()))
assert config.llm_provider(), 'LLM이 설정되지 않았습니다(.env). 에이전트 실행에 필요합니다.'

## 2. 보낼 샘플 콘텐츠 정의

실제 위키에 담길 법한 재사용 가능한 기술 지식/노하우 예시입니다. 자유롭게 편집하세요.

In [ ]:
MAIL_TARGET = '나 자신(로그인한 사용자)의 메일함으로'  # 예: 'alice@contoso.com 에게'
TEAMS_TARGET = '나 자신과의 1:1 채팅(없으면 새로 생성)'  # 예: "'Platform' 팀의 'General' 채널"

SAMPLE_EMAILS = [
    {
        'subject': '[노하우] Postgres 커넥션 풀 튜닝 정리',
        'body': (
            'PgBouncer를 transaction 모드로 두고 app 쪽 pool_size는 CPU 코어수*2로 제한. '
            'idle_in_transaction_session_timeout=30s 로 좀비 커넥션 방지. '
            'max_connections를 무작정 늘리기보다 PgBouncer로 멀티플렉싱하는 게 안정적이었음.'
        ),
    },
    {
        'subject': '[트러블슈팅] k8s 파드 CrashLoopBackOff 원인 추적법',
        'body': (
            'kubectl describe pod로 이벤트 확인 -> kubectl logs --previous로 직전 컨테이너 로그. '
            'OOMKilled면 requests/limits 재조정. readinessProbe 타임아웃이 잦으면 initialDelaySeconds 상향.'
        ),
    },
]

SAMPLE_TEAMS_MESSAGES = [
    ('[팁] GitHub Actions 캐시 히트율 올리기: setup-node의 cache: npm 대신 actions/cache로 '
     'node_modules + ~/.npm 둘 다 키에 lockfile 해시를 포함시키면 히트율이 크게 올라감.'),
    ('[결정사항] 사내 로그 표준: 구조적 로깅(JSON) + trace_id 필수. 레벨은 ERROR/WARN/INFO만 사용, '
     'DEBUG는 로컬 전용. 마이그레이션 가이드는 위키 참조.'),
]

print('메일', len(SAMPLE_EMAILS), '건, Teams', len(SAMPLE_TEAMS_MESSAGES), '건 준비됨')

## 3. 메일 샘플 전송 (Mail MCP)

In [ ]:
from pipeline import agent

if 'mail' in tokens:
    lines = [f"- 제목: {e['subject']}\n  본문: {e['body']}" for e in SAMPLE_EMAILS]
    instruction = (
        f'다음 이메일들을 각각 별도의 메일로 {MAIL_TARGET} 보내줘. '
        '제목과 본문을 그대로 사용하고, 전송 후 각 메일의 결과(성공/ID)를 요약해줘.\n\n'
        + '\n'.join(lines)
    )
    result = await agent.run_agent({'mail': tokens['mail']}, instruction)
    print(result['answer'])
    if result['source_errors']:
        print('오류:', result['source_errors'])
else:
    print('mail 토큰이 없어 건너뜀')

## 4. Teams 샘플 메시지 게시 (Teams MCP)

In [ ]:
if 'teams' in tokens:
    lines = [f'- {m}' for m in SAMPLE_TEAMS_MESSAGES]
    instruction = (
        f'다음 메시지들을 {TEAMS_TARGET} 각각 게시해줘. '
        '대상을 먼저 조회해서 실제 ID로 게시하고, 게시 결과를 요약해줘.\n\n'
        + '\n'.join(lines)
    )
    result = await agent.run_agent({'teams': tokens['teams']}, instruction)
    print(result['answer'])
    if result['source_errors']:
        print('오류:', result['source_errors'])
else:
    print('teams 토큰이 없어 건너뜀')

✅ 샘플이 전송되었습니다. 인덱싱 지연이 있을 수 있으니 잠시 후 **03_fetch_data**에서 조회하세요.